# Tool-Using Agent with Real API Integrations
### Multi-API Orchestration with OpenAI Tool Calling, Tavily, Open-Meteo, and SerpAPI

This notebook builds a **runnable tool-using agent** that can plan a trip using **real APIs**.

It demonstrates:

- **OpenAI tool calling** for structured function invocation
- **Tavily** for live web search
- **Open-Meteo** for weather forecasts
- **SerpAPI** for:
  - Google Flights results
  - Google Hotels results
- A practical **agent orchestration loop** with:
  - tracing
  - safe JSON handling
  - retry logic
  - support for multiple tool calls in one turn


This notebook performs **availability and discovery only**. It does **not** perform booking, payment, or any irreversible action.


## 0. Setup Notes and Required API Keys

You will need the following environment variables:

- `OPENAI_API_KEY`
- `TAVILY_API_KEY`
- `SERPAPI_KEY`

Recommended local `.env` file:

```bash
OPENAI_API_KEY="..."
TAVILY_API_KEY="..."
SERPAPI_KEY="..."
```

Also add `.env` to `.gitignore` before pushing to GitHub.

## Why tool use matters here

The model does **not** directly execute code or query external systems.

Instead:

1. The model decides that external data is needed
2. It emits a **structured tool call**
3. The Python runtime executes that call
4. The result is returned to the model
5. The model continues reasoning with grounded data

This separation of **reasoning** and **execution** is what makes tool-using agents robust, inspectable, and safe.


## 1. Install Dependencies

We install the libraries required for:

- OpenAI model calls
- Tavily web search
- environment variable loading
- HTTP requests to Open-Meteo and SerpAPI



In [ ]:
!pip -q install openai requests python-dotenv tavily-python

## 2. Import Libraries

We import:

- standard utilities like `os`, `json`, `time`, and `logging`
- type hints for cleaner function signatures
- `requests` for direct REST API calls
- `dotenv` to load secrets from a local `.env`
- `OpenAI` and `TavilyClient` SDKs

We use direct HTTP requests for Open-Meteo and SerpAPI because they are simple REST APIs and this keeps the notebook transparent for learning purposes.


In [ ]:
from google.colab import userdata
import os
import json
import time
import logging
from typing import Any, Dict, List, Optional, Tuple

import requests
from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient


## 3. Configure Logging and Initialize Clients

A tool-using agent is easier to debug when every action is observable.

We configure logging to show:

- orchestration steps
- requested tools
- arguments
- retries
- failures

Then we load environment variables and initialize the API clients.

If a key is missing, the notebook still loads, but the related tool will return a clear error when called. That makes setup issues much easier to diagnose.


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger("tool-agent")

load_dotenv()

os.environ["OPENAI_API_KEY"] = userdata.get("gpt_chat_key")
os.environ["TAVILY_API_KEY"] = userdata.get("tavily_api_key")
os.environ["SERPAPI_KEY"] = userdata.get("serp_api_key")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
SERPAPI_KEY = os.getenv("SERPAPI_KEY")

openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
tavily_client = TavilyClient(api_key=TAVILY_API_KEY) if TAVILY_API_KEY else None

logger.info("Clients initialized. Missing keys will surface when a corresponding tool is called.")

## 4. Utility Functions for Reliability and Token Control

Real tools can return large payloads. If we pass everything back to the model, we may waste tokens and slow the loop.

So we implement helper utilities for:

- safe JSON serialization
- truncating oversized tool results before sending them back to the model
- retrying unstable HTTP calls

These utilities are lightweight, but they reflect good production habits.


In [ ]:
def safe_json_dumps(obj: Any, max_len: int = 200_000) -> str:
    """Safely serialize an object to JSON and hard-truncate extreme payload sizes."""
    try:
        s = json.dumps(obj, ensure_ascii=False, default=str)
    except Exception as e:
        s = json.dumps({"error": f"JSON serialization failed: {e}", "repr": repr(obj)}, ensure_ascii=False)
    if len(s) > max_len:
        return s[:max_len] + f"...[truncated {len(s)-max_len} chars]"
    return s

def truncate_for_model(obj: Any, max_chars: int = 18_000) -> str:
    """Trim tool outputs before feeding them back into the model context."""
    s = safe_json_dumps(obj)
    if len(s) <= max_chars:
        return s
    return s[:max_chars] + f"...[truncated {len(s)-max_chars} chars]"

def request_json_with_retries(
    method: str,
    url: str,
    *,
    params: Optional[Dict[str, Any]] = None,
    headers: Optional[Dict[str, str]] = None,
    timeout: int = 30,
    max_retries: int = 3,
    backoff_s: float = 1.0
) -> Dict[str, Any]:
    """Retry simple HTTP JSON requests on transient failures."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.request(
                method,
                url,
                params=params,
                headers=headers,
                timeout=timeout
            )
            if response.status_code >= 500:
                raise RuntimeError(f"Server error {response.status_code}: {response.text[:300]}")
            response.raise_for_status()
            return response.json()
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                sleep = backoff_s * attempt
                logger.warning(f"HTTP attempt {attempt} failed: {e}. Retrying in {sleep:.1f}s...")
                time.sleep(sleep)
            else:
                logger.error(f"HTTP request failed after {max_retries} attempts: {e}")
    return {"error": str(last_err)}

## 5. Tool 1: Weather Forecast with Open-Meteo

This tool performs two steps:

1. Resolve a city name to coordinates using the geocoding endpoint
2. Request a daily forecast using the forecast endpoint

We return a compact structured result containing:

- resolved location
- daily dates
- min and max temperatures
- max precipitation probability

This gives the model enough information to reason without flooding the context with unnecessary fields.


In [ ]:
def get_weather(city: str) -> Dict[str, Any]:
    logger.info(f"[tool] get_weather(city={city})")

    geo = request_json_with_retries(
        "GET",
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1, "language": "en", "format": "json"},
        max_retries=3
    )
    if "error" in geo:
        return {"error": "Geocoding failed", "details": geo["error"]}

    results = geo.get("results") or []
    if not results:
        return {"error": "City not found", "city": city}

    lat = results[0]["latitude"]
    lon = results[0]["longitude"]
    resolved_name = results[0].get("name")
    country = results[0].get("country")

    forecast = request_json_with_retries(
        "GET",
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
            "timezone": "auto"
        },
        max_retries=3
    )
    if "error" in forecast:
        return {"error": "Forecast failed", "details": forecast["error"]}

    daily = forecast.get("daily", {})
    return {
        "location": {
            "input_city": city,
            "resolved_name": resolved_name,
            "country": country,
            "lat": lat,
            "lon": lon
        },
        "daily": {
            "time": daily.get("time", []),
            "t_max_c": daily.get("temperature_2m_max", []),
            "t_min_c": daily.get("temperature_2m_min", []),
            "precip_prob_max": daily.get("precipitation_probability_max", [])
        }
    }

## 6. Tool 2: Web Search with Tavily

Travel requests often need fresh context such as:

- destination overviews
- current local advice
- event context
- seasonal conditions
- practical travel notes

The web search tool grounds the agent in current information.

We keep the output intentionally compact by returning only the most useful fields:

- title
- url
- snippet/content


In [ ]:
def web_search(query: str, max_results: int = 5) -> Dict[str, Any]:
    logger.info(f"[tool] web_search(query={query}, max_results={max_results})")

    if tavily_client is None:
        return {"error": "Tavily client not initialized. Missing TAVILY_API_KEY."}

    try:
        res = tavily_client.search(query=query, max_results=max_results)
        results = res.get("results", [])
        compact = [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "content": r.get("content")
            }
            for r in results
        ]
        return {"query": query, "results": compact}
    except Exception as e:
        return {"error": f"Tavily search failed: {e}"}

## 7. Tool 3: Flight Search with SerpAPI (Google Flights)

This tool queries SerpAPI using the `google_flights` engine.

Inputs:
- `origin`: IATA code such as `MAD`
- `destination`: IATA code such as `BER`
- `outbound_date`: ISO date in `YYYY-MM-DD`

Important notes:

- SerpAPI mirrors Google travel-style results
- the exact returned structure may evolve over time
- this notebook returns the full JSON payload, then truncates it for the model if needed
- this is **search only**, not booking


In [ ]:
def search_flights(origin: str, destination: str, outbound_date: str) -> dict:
    logger.info(
        f"[tool] search_flights(origin={origin}, destination={destination}, outbound_date={outbound_date})"
    )

    if not SERPAPI_KEY:
        return {"error": "Missing SERPAPI_KEY."}

    params = {
        "engine": "google_flights",
        "departure_id": origin,
        "arrival_id": destination,
        "outbound_date": outbound_date,
        "type": "2",          # one-way
        "hl": "en",
        "gl": "us",
        "currency": "USD",
        "api_key": SERPAPI_KEY,
    }

    data = request_json_with_retries(
        "GET",
        "https://serpapi.com/search",
        params=params,
        max_retries=3,
    )

    if "error" in data:
        return {"error": "SerpAPI flight search failed", "details": data["error"], "params": params}

    return {
        "query": {
            "origin": origin,
            "destination": destination,
            "outbound_date": outbound_date,
            "type": "one_way",
        },
        "results": data,
    }

## 8. Tool 4: Hotel Search with SerpAPI (Google Hotels)

This tool queries SerpAPI using the `google_hotels` engine.

Input:
- `city`: city or destination query, such as `Berlin`

Output:
- hotel results returned by the API

Again, the output structure can evolve. For teaching purposes, that is actually useful because students can inspect real-world tool payloads and see why robust parsing and truncation matter in agent systems.


In [ ]:
def search_hotels(city: str, check_in_date: str, check_out_date: str) -> dict:
    logger.info(
        f"[tool] search_hotels(city={city}, check_in_date={check_in_date}, check_out_date={check_out_date})"
    )

    if not SERPAPI_KEY:
        return {"error": "Missing SERPAPI_KEY."}

    params = {
        "engine": "google_hotels",
        "q": city,
        "check_in_date": check_in_date,
        "check_out_date": check_out_date,
        "adults": "2",
        "hl": "en",
        "gl": "us",
        "currency": "USD",
        "api_key": SERPAPI_KEY,
    }

    data = request_json_with_retries(
        "GET",
        "https://serpapi.com/search",
        params=params,
        max_retries=3,
    )

    if "error" in data:
        return {"error": "SerpAPI hotel search failed", "details": data["error"], "params": params}

    return {
        "query": {
            "city": city,
            "check_in_date": check_in_date,
            "check_out_date": check_out_date,
        },
        "results": data,
    }

## 9. Define OpenAI Tool Schemas

We now expose the Python functions as callable tools.

Each tool schema includes:

- a clear name
- a concise description
- a JSON schema for arguments
- required fields

This matters because the model uses these schemas to decide:

- whether a tool is needed
- which tool to call
- how to format arguments

Better schemas usually mean better tool behavior.


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get a compact multi-day weather forecast for a city using Open-Meteo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, for example Berlin"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for up-to-date information and return top results.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_flights",
            "description": "Search Google Flights results through SerpAPI using origin and destination IATA codes and an outbound date. Search only, no booking.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {
                        "type": "string",
                        "description": "Origin IATA airport code, for example MAD"
                    },
                    "destination": {
                        "type": "string",
                        "description": "Destination IATA airport code, for example BER"
                    },
                    "outbound_date": {
                        "type": "string",
                        "description": "Outbound date in YYYY-MM-DD format"
                    }
                },
                "required": ["origin", "destination", "outbound_date"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_hotels",
            "description": "Search Google Hotels results through SerpAPI for a city and stay dates. Search only, no booking.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "check_in_date": {"type": "string", "description": "YYYY-MM-DD"},
                    "check_out_date": {"type": "string", "description": "YYYY-MM-DD"}
                },
                "required": ["city", "check_in_date", "check_out_date"]
            }
        }
    }
]

## 10. Tool Dispatcher

The dispatcher maps model-requested tool calls to actual Python functions.

This is the right place to add, in more advanced systems:

- permission checks
- rate limits
- audit trails
- caching
- argument normalization
- output post-processing

For now, we keep it simple and robust.

We return two versions of the result:

1. the raw Python object
2. a truncated JSON string safe to send back into the model context


In [ ]:
def execute_tool(name: str, args: Dict[str, Any]) -> Tuple[Dict[str, Any], str]:
    """Execute one tool call and return both the raw result and a model-safe serialized form."""
    try:
        if name == "get_weather":
            raw = get_weather(**args)
        elif name == "web_search":
            raw = web_search(**args)
        elif name == "search_flights":
            raw = search_flights(**args)
        elif name == "search_hotels":
            raw = search_hotels(**args)
        else:
            raw = {"error": f"Unknown tool: {name}"}
    except Exception as e:
        raw = {"error": f"Tool execution crashed: {e}", "tool": name, "args": args}

    return raw, truncate_for_model(raw)

## 11. System Prompt

The system prompt defines the agent's operating policy.

We want the agent to:

- use tools whenever live or current data is required
- avoid hallucinating travel availability
- stay within the demo constraints
- produce a clear final answer

This prompt is not magic, but it strongly improves consistency in tool-using workflows.


In [ ]:
CURRENT_DATE = "2026-03-16"

In [ ]:
SYSTEM_PROMPT = f"""You are a careful travel-planning assistant with tool access.

Rules:
- Use tools whenever live data is needed.
- Do not invent flight or hotel availability.
- Do not attempt booking, payment, or any irreversible action.
- For flight searches, always use a future outbound_date in YYYY-MM-DD format.
- For hotel searches, always use future check_in_date and check_out_date in YYYY-MM-DD format.
- If the user says 'next week', convert it relative to Today (make sure to retrieve today's date using web search).
- Never use dates in the past.
"""

## 12. Agent Orchestration Loop

This is the core execution loop.

Process:

1. Send the conversation and tool definitions to the model
2. Check whether the model requested tools
3. Execute all requested tools
4. Append the tool outputs
5. Repeat until a final answer is produced

This implementation supports:

- multiple tool calls in one model turn
- execution tracing
- maximum step control to avoid loops

This is the minimal orchestration pattern that later scales into more advanced agent architectures.


In [ ]:
def run_agent(
    user_input: str,
    *,
    model: str = "gpt-4o-mini",
    max_steps: int = 8,
    trace: bool = True
) -> Dict[str, Any]:
    if openai_client is None:
        return {"error": "OpenAI client not initialized. Missing OPENAI_API_KEY."}

    messages: List[Dict[str, Any]] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    tool_trace: List[Dict[str, Any]] = []

    for step in range(1, max_steps + 1):
        if trace:
            logger.info(f"=== Agent step {step}/{max_steps} ===")

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        if getattr(msg, "tool_calls", None):
            messages.append({
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": msg.tool_calls
            })

            for tc in msg.tool_calls:
                name = tc.function.name
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except Exception:
                    args = {"_raw_arguments": tc.function.arguments}

                if trace:
                    logger.info(f"Tool requested: {name} | args={args}")

                raw_result, serialized_result = execute_tool(name, args)

                tool_trace.append({
                    "step": step,
                    "tool": name,
                    "args": args,
                    "result_preview": serialized_result[:5000]
                })

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": serialized_result
                })

            continue

        final_text = msg.content or ""
        if trace:
            logger.info("=== Agent finished (no more tool calls) ===")

        return {
            "answer": final_text,
            "tool_trace": tool_trace,
            "messages": messages
        }

    return {
        "error": f"Max steps ({max_steps}) reached without a final answer.",
        "tool_trace": tool_trace,
        "messages": messages
    }

## 13. Run a Demo Query

We now test the full agent with a realistic trip-planning request.

The prompt asks for:

- flights from Madrid to Berlin
- hotel options in Berlin
- weather in Berlin
- a synthesized travel recommendation

This is a good example because it forces the model to orchestrate multiple tools rather than relying on static knowledge.


In [ ]:
print(search_flights("MAD", "BER", "2026-03-23"))
print(search_hotels("Berlin", "2026-03-23", "2026-03-26"))

In [ ]:
demo_query = """Plan a 3-day trip to Berlin next week.
Include:
1) a quick weather snapshot,
2) flight options from MAD to BER,
3) hotel options in Berlin,
and then summarize a suggested plan.
No bookings."""

result = run_agent(demo_query, trace=True)
print(result.get("answer") or result)

## 14. Inspect the Tool Trace

One of the most important properties of a good agent system is **observability**.

This cell prints the execution trace so you can inspect:

- which tools were called
- in what order
- with what arguments
- what kind of results came back

This is especially useful when teaching, debugging, or comparing orchestration behaviors across prompts.


In [ ]:
if "tool_trace" in result:
    for i, t in enumerate(result["tool_trace"], start=1):
        print(f"\n--- Tool Call {i} ---")
        print("step:", t["step"])
        print("tool:", t["tool"])
        print("args:", t["args"])
        print("result_preview:", t["result_preview"][:1000], "...")

## 15. Suggested Next Improvements

This notebook is already runnable and demo-ready, but there are several strong upgrades you can add next:

### Better date resolution
Add a helper tool to convert phrases like:
- next week
- next Friday
- in 3 days

into explicit ISO dates.

### Airport and city code resolution
Add a lookup tool so the model does not need to infer codes like:
- MAD
- BER

### Structured parsing of SerpAPI outputs
Right now we pass the raw SerpAPI payload through the loop. A useful improvement is to normalize it into a smaller travel schema.

### Caching
Avoid repeated API calls for identical queries during experimentation.

### Evaluation
Create test prompts and log:
- tool usage accuracy
- missing data cases
- final answer quality

But even with those future upgrades, the architecture you built here is already the essential core of a real tool-using agent.
